# Вариант 15

## Прямая задача

\
max z = 3x_1 - x_2 + 2x_3 + x_4


при ограничениях:

\
\begin{cases}
x_1 - x_2 + 2x_3 + x_4 \le 8\\
3x_2 + x_3 \le 7\\
4x_1 + x_2 + 3x_3 + 2x_4 \le 15\\
x_1, x_2, x_3, x_4 \ge 0
\end{cases}

Матрицы:

\
c = \begin{pmatrix} 3 \\ -1 \\ 2 \\ 1 \end{pmatrix},  
A = \begin{pmatrix}
1 & -1 & 2 & 1\\
0 & 3 & 1 & 0\\
4 & 1 & 3 & 2
\end{pmatrix},  
b = \begin{pmatrix} 8 \\ 7 \\ 15 \end{pmatrix}


## Двойственная задача

\
min w = 8y_1 + 7y_2 + 15y_3


при ограничениях:

\
\begin{cases}
y_1 + 4y_3 \ge 3\\
-y_1 + 3y_2 + y_3 \ge -1\\
2y_1 + y_2 + 3y_3 \ge 2\\
y_1 + 2y_3 \ge 1\\
y_1, y_2, y_3 \ge 0
\end{cases}

In [ ]:
import numpy as np
from scipy.optimize import linprog

# Прямая задача
c = np.array([3., -1., 2., 1.])

A = np.array([
    [1., -1., 2., 1.],
    [0.,  3., 1., 0.],
    [4.,  1., 3., 2.]
])

b = np.array([8., 7., 15.])

# scipy решает минимизацию, поэтому для max берём -c
res_primal = linprog(
    c=-c,
    A_ub=A,
    b_ub=b,
    bounds=[(0, None)] * 4,
    method='highs'
)

x = res_primal.x
z_max = -res_primal.fun

print("Прямая задача")
print("success:", res_primal.success)
print("x =", x)
print("z_max =", z_max)

# Двойственная задача
# min b^T y
# A^T y >= c
# y >= 0
# scipy: A_ub y <= b_ub, поэтому -A^T y <= -c
res_dual = linprog(
    c=b,
    A_ub=-A.T,
    b_ub=-c,
    bounds=[(0, None)] * 3,
    method='highs'
)

y = res_dual.x
w_min = res_dual.fun

print("\nДвойственная задача")
print("success:", res_dual.success)
print("y =", y)
print("w_min =", w_min)

Прямая задача
success: True
x = [3.75 0.   0.   0.  ]
z_max = 11.25

Двойственная задача
success: True
y = [0.   0.   0.75]
w_min = 11.25


In [3]:
# Проверка допустимости решения прямой задачи
x_opt = res_primal.x

print("Проверка допустимости прямой задачи")
print("-" * 50)

for i in range(len(b)):
    lhs = A[i] @ x_opt
    print(f"Ограничение {i+1}: {lhs:.6f} <= {b[i]}  ->",
          "OK" if lhs <= b[i] + 1e-9 else "НАРУШЕНО")

print("Неотрицательность x:",
      "OK" if all(xi >= -1e-9 for xi in x_opt) else "НАРУШЕНО")


# Проверка теоремы сильной двойственности
print("\nПроверка теоремы двойственности")
print("-" * 50)
print(f"z_max = {z_max:.6f}")
print(f"w_min = {w_min:.6f}")
print("z_max == w_min ->", np.isclose(z_max, w_min))


# Проверка условий дополняющей нежёсткости
y_opt = res_dual.x

# Запасы в ограничениях прямой задачи: b - A x
primal_slack = b - A @ x_opt
# Запасы в ограничениях двойственной: A^T y - c
dual_slack = A.T @ y_opt - c

print("\nУсловия дополняющей нежёсткости")
print("-" * 50)

print("Запасы прямой задачи (b - A x):",
      [round(float(v), 6) for v in primal_slack])
print("Запасы двойственной (A^T y - c):",
      [round(float(v), 6) for v in dual_slack])

print("y_i * primal_slack_i =",
      [round(float(v), 6) for v in y_opt * primal_slack])
print("x_j * dual_slack_j =",
      [round(float(v), 6) for v in x_opt * dual_slack])

ok1 = np.allclose(y_opt * primal_slack, 0, atol=1e-9)
ok2 = np.allclose(x_opt * dual_slack, 0, atol=1e-9)
print("Условия дополняющей нежёсткости выполняются:",
      ok1 and ok2)

Проверка допустимости прямой задачи
--------------------------------------------------
Ограничение 1: 3.750000 <= 8.0  -> OK
Ограничение 2: 0.000000 <= 7.0  -> OK
Ограничение 3: 15.000000 <= 15.0  -> OK
Неотрицательность x: OK

Проверка теоремы двойственности
--------------------------------------------------
z_max = 11.250000
w_min = 11.250000
z_max == w_min -> True

Условия дополняющей нежёсткости
--------------------------------------------------
Запасы прямой задачи (b - A x): [4.25, 7.0, 0.0]
Запасы двойственной (A^T y - c): [0.0, 1.75, 0.25, 0.5]
y_i * primal_slack_i = [0.0, 0.0, 0.0]
x_j * dual_slack_j = [0.0, 0.0, 0.0, 0.0]
Условия дополняющей нежёсткости выполняются: True



**Проверка решения**

Найденное решение $x^* = (3.75,\ 0,\ 0,\ 0)$ подставим в ограничения:

$$3.75 \le 8,\quad 0 \le 7,\quad 15 \le 15$$

Все ограничения и условия неотрицательности выполняются — решение **допустимо**.

Значения целевых функций совпадают:

$$z_{\max} = w_{\min} = 11.25$$

Значит, **теорема сильной двойственности выполняется**.

Проверка условий дополняющей нежёсткости: все произведения $y_i \cdot \text{slack}_i$ и $x_j \cdot \text{slack}_j$ равны нулю — условия **выполнены**, что подтверждает оптимальность решений.

## Ответ

Оптимальное решение прямой задачи:

$$x_1 = 3.75,\quad x_2 = 0,\quad x_3 = 0,\quad x_4 = 0$$

$$z_{\max} = 3 \cdot 3.75 - 1 \cdot 0 + 2 \cdot 0 + 1 \cdot 0 = 11.25$$

Оптимальное решение двойственной задачи:

$$y_1 = 0,\quad y_2 = 0,\quad y_3 = 0.75$$

$$w_{\min} = 8 \cdot 0 + 7 \cdot 0 + 15 \cdot 0.75 = 11.25$$

Проверка по теореме двойственности:

$$z_{\max} = w_{\min} = 11.25$$

Следовательно, решение верно.
